# Exercícios práticos — LangChain, LangGraph, HITL e Gradio

Caderno baseado no guia `complete_guide.md`. Resolva os exercícios em ordem. Cada enunciado é seguido por uma célula de código Python vazia.

**Ambiente:** use as dependências e credenciais já configuradas no seu projeto. Exercícios que chamam modelos exigem um modelo compatível com tool calling e credenciais válidas. Não exponha chaves em células ou saídas.

**Dependências:** os exercícios marcados como independentes podem ser feitos isoladamente; os marcados como reutilização ou desafio integrador dependem de componentes anteriores. Quando um exercício pedir análise conceitual, registre sua análise em comentários ou em estruturas Python na célula de resposta.

## Preparação do ambiente

Execute esta célula a partir do projeto para instalar ou atualizar as bibliotecas usadas no guia. Reinicie o kernel se o ambiente já estava aberto.

In [2]:
!uv add --upgrade langchain langgraph langchain-openai gradio

Resolved 240 packages in 365ms                                       
Checked 221 packages in 29ms


In [3]:
import os

from dotenv import load_dotenv
from langchain_ollama import ChatOllama

load_dotenv(override=True)

model = os.getenv("OLLAMA_MODEL") or "batiai/gemma4-e2b:q4"
chat_model = ChatOllama(model=model)

print(
    chat_model.invoke(input="Olá, explica o gato de Shrodinger sucintamente.").content
)

O Gato de Schrödinger é um **experimento mental** proposto pelo físico Erwin Schrödinger em 1935 para ilustrar o paradoxo da **mecânica quântica**.

**Em resumo:**

1. **A Ideia:** Imagine um gato colocado em uma caixa hermeticamente fechada com um mecanismo que depende do decaimento de um átomo radioativo. Se o átomo decair, um mecanismo libera veneno e o gato morre; se não decair, o gato vive.
2. **O Paradoxo Quântico:** Na mecânica quântica, antes de a caixa ser aberta (ou seja, antes de a medição), o átomo existe em uma "superposição" – ele está simultaneamente no estado de ter decaído *e* não ter decaído.
3. **A Conclusão Paradoxal:** Se o estado do gato depende diretamente do estado quântico do átomo, então, enquanto a caixa estiver fechada, o gato deve estar em uma **superposição** – ele estaria simultaneamente **vivo e morto**.

**O Ponto Principal:** O experimento não busca demonstrar que gatos podem ser simultaneamente vivos e mortos, mas sim expor o absurdo de aplicar as reg

## 1. Agente, modelo, mensagens e ciclo de execução

### Exercício 1 — Criar e invocar um agente mínimo

**Independente.** Importe `create_agent`, declare uma constante para o identificador de um modelo com suporte a tool calling e crie um agente sem tools. Faça uma invocação com uma mensagem do usuário e extraia o conteúdo da resposta final. Não implemente um loop manual de chamadas ao modelo.

In [4]:
from langchain.agents import create_agent

agent = create_agent(model=chat_model)

response = agent.invoke({"messages": [{"role": "user", "content": "Olá"}]})
response

{'messages': [HumanMessage(content='Olá', additional_kwargs={}, response_metadata={}, id='ec382e80-7c36-45e8-8004-fe649d8ce8c5'),
  AIMessage(content='Olá! Como posso ajudar você hoje? 😊', additional_kwargs={}, response_metadata={'model': 'batiai/gemma4-e2b:q4', 'created_at': '2026-11-14T23:44:23.929981518Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3011527368, 'load_duration': 511793400, 'prompt_eval_count': 24, 'prompt_eval_duration': 32890000, 'eval_count': 228, 'eval_duration': 2459520000, 'logprobs': None, 'model_name': 'batiai/gemma4-e2b:q4', 'model_provider': 'ollama'}, id='lc_run--01a1dc72-9bab-72f9-b621-d4426ea55a5d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_tokens': 228, 'total_tokens': 252})]}

### Exercício 2 — Inspecionar o ciclo de mensagens

**Independente.** Dada uma lista contendo uma mensagem humana, uma `AIMessage` com `tool_calls`, uma `ToolMessage` e uma resposta final `AIMessage`, escreva uma função que classifique cada etapa e explique no retorno qual componente produziu a mensagem e para onde o fluxo segue.

In [5]:
messages = [
    {"role": "user", "content": "Qual é a capital de Portugal?"},
    {
        "role": "assistant",
        "tool_calls": [
            {"id": "call_1", "name": "buscar_capital", "args": {"pais": "Portugal"}}
        ],
    },
    {"role": "tool", "tool_call_id": "call_1", "content": "Lisboa"},
    {"role": "assistant", "content": "A capital de Portugal é Lisboa."},
]

### Exercício 3 — Comparar entrada em formato de dicionário e mensagens tipadas

**Independente.** Construa a mesma entrada de conversa primeiro com dicionários `role`/`content` e depois com classes de mensagem do LangChain. Faça uma invocação em cada formato e compare a estrutura relevante do estado retornado.

In [6]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

messages = [
    HumanMessage(content="Qual é a capital de Portugal?"),
    AIMessage(
        content="",
        tool_calls=[
            {
                "name": "buscar_capital",
                "args": {"pais": "Portugal"},
                "id": "call_123",
            }
        ],
    ),
    ToolMessage(
        name="buscar_capital",
        content="Lisboa",
        tool_call_id="call_123",
    ),
    AIMessage(content="A capital de Portugal é Lisboa."),
]

## 2. Tools: definição, registro, argumentos e resultados

### Exercício 4 — Implementar e registrar uma tool determinística

**Independente.** Defina com `@tool` uma função que converta uma temperatura de Celsius para Fahrenheit. Use anotações de tipo e uma docstring que deixe claros o propósito e os argumentos. Registre a tool em `create_agent` e prepare uma invocação que peça ao modelo para usá-la.

In [7]:
from pprint import pprint

from langchain.tools import tool


@tool
def convert_to_fahrenheit(celsius: float):
    """Convert temperature from Celsius to Fahrenheit

    Args:
        celsius (float): temperature in Celsius

    Returns:
        int: temperature in Fahrenheit, rounded
    """
    return round(celsius * 9 / 5 + 32)


agent = create_agent(model=chat_model, tools=[convert_to_fahrenheit])

response = agent.invoke(
    {"messages": [{"role": "user", "content": "Converta 55 Celsius em Fahrenheit"}]}
)

pprint(response)

{'messages': [HumanMessage(content='Converta 55 Celsius em Fahrenheit', additional_kwargs={}, response_metadata={}, id='babc3242-f8b4-49d0-9100-e1b3db099b73'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'batiai/gemma4-e2b:q4', 'created_at': '2026-11-14T23:44:26.276310291Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2265464574, 'load_duration': 450225788, 'prompt_eval_count': 104, 'prompt_eval_duration': 59870000, 'eval_count': 176, 'eval_duration': 1689028000, 'logprobs': None, 'model_name': 'batiai/gemma4-e2b:q4', 'model_provider': 'ollama'}, id='lc_run--01a1dc72-a7c8-7447-a46e-b2b744cc8a9c-0', tool_calls=[{'name': 'convert_to_fahrenheit', 'args': {'celsius': 55}, 'id': 'b61cbf50-93a0-470c-944a-e2684d762bdc', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 104, 'output_tokens': 176, 'total_tokens': 280}),
              ToolMessage(content='131', name='convert_to_fahrenheit', id='65fcde9a-6ce1-497a-a0

### Exercício 5 — Validar argumentos de uma tool

**Independente.** Implemente uma tool que recebe uma quantidade e um preço unitário, rejeita valores negativos com erro claro e retorna o subtotal arredondado a duas casas. Demonstre a validação da função diretamente e descreva como os argumentos tipados são apresentados ao modelo.

In [8]:
from langchain.tools import tool
from langchain_core.messages import HumanMessage


@tool
def subtotal(quantity: float, price: float):
    """_summary_

    Args:
        quantity (float): quantidade
        price (float): preço unitário

    Returns:
        Optional[float, str]: _description_
    """
    if price < 0:
        return "Erro: preço negativo inválido."
    return round(quantity * price, 2)


agent = create_agent(model=chat_model, tools=[subtotal])

response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Qual total: 5 itens a R$ 8,22 cada? E se tivermos 2 itens a -1,55 cada?"
            )
        ]
    }
)

print(response["messages"][-1].content)

O primeiro cálculo é:
*   5 itens a R$ 8,22 cada: **R$ 41,10** (5 * 8,22 = 41,10)

Para o segundo cálculo:
*   Não foi possível calcular o total para os 2 itens a -R$ 1,55 cada, pois o preço unitário não pode ser negativo.


### Exercício 6 — Reconstruir o ciclo de uma chamada

**Independente.** Escreva uma função que receba uma sequência de mensagens e devolva, para cada tool call, seu nome, argumentos, `tool_call_id` e o `ToolMessage` correspondente. Trate o caso em que o resultado da tool não está presente ainda.

### Exercício 7 — Distinguir execução pelo agente de chamada manual

**Independente.** Implemente uma pequena análise que, dado um trecho de fluxo com `agent.invoke`, `AIMessage.tool_calls` e `ToolMessage`, identifique quem solicita a tool, quem a executa e como o resultado retorna ao modelo. Não invoque manualmente a tool após a resposta do agente.

## 3. AgentState, Context, ToolRuntime e memória

### Exercício 8 — Ler o estado com ToolRuntime

**Independente.** Defina uma tool que receba `ToolRuntime` injetado e retorne a quantidade de mensagens em `runtime.state`. Mostre como registrar essa tool no agente e por que o modelo não deve fornecer o argumento `runtime`.

In [27]:
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

@tool
def contar_mensagens(runtime: ToolRuntime) -> int:
    """Conta as mensagens da interação, considerando a atual

    Args:
        runtime (ToolRuntime): ToolRuntime object

    Returns:
        int: quantidade de menagens trocadas
    """
    return len(runtime.state.get("messages", []))


agent = create_agent(model=chat_model, tools=[contar_mensagens])

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="Olá"),
            AIMessage(content="Olá! O que deseja?"),
            HumanMessage(content="Quanta mensagens trocamos nessa interação?"),
        ]
    }
)

response.get("messages", [])[-1].content

'Nesta interação, trocamos 4 mensagens.'

### Exercício 9 — Disponibilizar Context por invocação

**Independente.** Crie um `Context` com identificador do usuário e nível de acesso. Registre seu tipo com `context_schema`, passe um objeto Context na invocação e implemente uma tool que consulte esse dado por meio de `ToolRuntime[Context]`.

In [37]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.messages import HumanMessage, SystemMessage
from langchain.tools import tool, ToolRuntime


@dataclass
class Context:
    user_id: int
    access_level: int


@tool
def inspecionar_usuario(context: ToolRuntime[Context]):
    """Inspeciona o usuário e seu nível de acesso

    Args:
        context (ToolRuntime[Context]): contexto
    """
    print("Resultado da inspeção: ", context.context)


agent = create_agent(
    model=chat_model, context_schema=Context, tools=[inspecionar_usuario]
)

response = agent.invoke(
    {
        "messages": [
            SystemMessage(
                content="Você é um inspetor disciplinado. Sua tarefa é inspecionar o usuário em todas as interações que ele faça, sem deixar nenhuma interação sem inspeção."
            ),
            HumanMessage(content="Olá, parça."),
        ]
    },
    context=Context(user_id=1, access_level=10),
)

response["messages"][-1].content

Resultado da inspeção:  Context(user_id=1, access_level=10)


'Olá. Como posso ajudá-lo hoje?'

### Exercício 10 — Separar estado, contexto e configuração

**Independente.** Implemente uma função que recebe os campos relevantes de uma execução e os classifica como estado de conversa, contexto da invocação, configuração/checkpoint ou armazenamento de longo prazo. Inclua `runtime.state`, `runtime.context`, `runtime.config` e `runtime.store` no caso analisado.

### Exercício 11 — Persistir histórico com Checkpointer

**Independente.** Crie um `InMemorySaver`, registre-o em `create_agent` e faça duas invocações usando o mesmo `thread_id`: uma que informe uma preferência fictícia e outra que a consulte. Repita a consulta com outro `thread_id` e compare o comportamento.

### Exercício 12 — Separar conversas por thread_id

**Reutiliza agente e checkpointer do exercício anterior.** Escreva uma função `make_config(thread_id)` e use-a para enviar mensagens a duas threads. Demonstre que cada thread recupera seu próprio histórico e explique o papel do checkpointer em cada invocação.

### Exercício 13 — Implementar memória longa no Store

**Independente.** Crie um `InMemoryStore` e duas tools para salvar e consultar uma preferência. Use `runtime.context.user_id` como parte do namespace e a chave informada como chave do item. Trate o resultado de consulta inexistente sem acessar `.value` de `None`.

### Exercício 14 — Integrar Store, Context e agente

**Reutiliza o Context e as tools do exercício anterior.** Registre `store`, `context_schema` e ambas as tools no agente. Implemente duas invocações em threads diferentes, mas com o mesmo usuário, para demonstrar que o histórico curto é por thread e a preferência persistida é compartilhada pelo namespace do usuário.

### Exercício 15 — Decidir quando a memória longa será acessada

**Independente.** Escreva um teste de integração que verifica que apenas configurar `store=...` não altera automaticamente uma resposta. Em seguida, registre tools de gravação e leitura e descreva no teste qual chamada solicitada pelo modelo efetivamente acessa o Store.

## 4. Middleware: hooks, handlers e ordem de execução

### Exercício 16 — Implementar hooks de nó

**Independente.** Implemente um `before_model` que registre a contagem de mensagens recebida em `AgentState` e retorne o valor apropriado para não alterar o estado. Registre-o em `middleware` e indique em que momento do ciclo ele roda.

### Exercício 17 — Implementar wrapper de tool

**Independente.** Implemente um `wrap_tool_call` que registre o nome da tool antes da chamada, invoque `handler(request)` uma única vez e registre o término. Retorne o resultado recebido pelo handler.

### Exercício 18 — Implementar wrapper de modelo

**Independente.** Implemente um `wrap_model_call` que conte as chamadas ao handler, invoque-o uma vez e devolva a resposta sem modificá-la. Explique com comentários onde a execução do modelo ocorre em relação ao wrapper.

### Exercício 19 — Rastrear ordem de hooks e wrappers

**Independente.** Dado um agente com `before_model`, `wrap_model_call`, `wrap_tool_call` e uma tool, produza um rastreador de eventos que registre a ordem observável para uma rodada em que o modelo chama uma tool. Separe hooks pontuais de wrappers que cercam handlers.

### Exercício 20 — Registrar middleware predefinido de sumarização

**Independente.** Configure `SummarizationMiddleware` com um modelo, limite por tokens e quantidade de mensagens a preservar. Registre-o junto a um middleware próprio e a um checkpointer. Escreva uma função que valide a configuração e explique por que o middleware atua sobre o histórico.

### Exercício 21 — Escolher middleware para o problema

**Independente.** Implemente uma função de decisão que associe cenários de logging pré-modelo, fallback/retry do modelo, logging de execução de tool, sumarização e limites de execução ao hook ou middleware apropriado. Inclua o racional para cada associação.

## 5. Human-in-the-Loop e retomada

### Exercício 22 — Definir política HITL por tool

**Independente.** Defina uma tool simulada para reservar um recurso e outra tool de consulta sem efeitos colaterais. Configure `HumanInTheLoopMiddleware` para permitir `approve`, `edit` e `reject` na reserva, deixar a consulta sem interrupção e usar `interrupt_mode=per_call`.

### Exercício 23 — Interpretar um interrupt

**Independente.** Escreva uma função que leia `result.interrupts`, extraia de cada entrada o `id`, nome da ação e argumentos, e retorne uma lista própria para revisão. Trate o resultado sem interrupções e deixe explícito que a tool protegida ainda não foi executada.

### Exercício 24 — Construir decisões HITL válidas

**Independente.** Implemente funções que montem os payloads de `approve`, `edit`, `reject` e `respond`. A edição deve identificar a tool e seus novos argumentos; a rejeição deve incluir uma mensagem. Descreva em qual tipo de tool `respond` substitui a execução.

### Exercício 25 — Retomar execução pelo mesmo checkpoint

**Reutiliza agente HITL com checkpointer.** Depois de capturar um interrupt, retome o fluxo via `Command(resume={interrupt_id: decision})`, usando o mesmo `thread_id` e configuração da invocação suspensa. Inspecione se o retorno contém resposta final ou novo interrupt.

### Exercício 26 — Processar uma interrupção por vez

**Reutiliza funções de extração e retomada anteriores.** Implemente um controlador que mantenha a lista de interrupts pendentes, apresente o primeiro, aplique a decisão por ID e processe novamente o resultado. Não assuma que um resume encerra o agente.

### Exercício 27 — Tratar múltiplas interrupções

**Reutiliza controlador anterior.** Implemente também a construção de um dicionário de decisões para vários IDs. Crie uma validação que confira se os IDs decididos pertencem às pendências atuais e explique como evitar reaplicar aprovações já processadas.

### Exercício 28 — Escolher modo batched ou per_call

**Independente.** Compare em uma função ou quadro estruturado o momento em que o modo `batched` agrupa revisões e o modo `per_call` intercepta cada chamada protegida. Associe cada modo a um fluxo e indique o efeito sobre chamadas múltiplas.

## 6. Gradio: componentes, estado, callbacks e eventos

### Exercício 29 — Criar o esqueleto de uma interface

**Independente.** Em `gr.Blocks`, defina `gr.State` para thread e pendências, um `gr.Chatbot(type=messages)`, um campo de mensagem e um painel de revisão inicialmente oculto com campos para argumentos JSON e motivo de rejeição. Não conecte callbacks ainda.

### Exercício 30 — Mapear inputs, parâmetros e outputs

**Independente.** Escreva um callback com dois componentes de entrada e dois valores de saída, depois conecte-o a um evento `.click()`. Faça um mapeamento explícito entre a ordem de `inputs`, parâmetros da função, valores retornados e ordem de `outputs`.

### Exercício 31 — Atualizar interface com múltiplos retornos

**Independente.** Implemente um callback que atualize simultaneamente o histórico do chatbot, o estado de pendências e a visibilidade do painel de revisão. Inclua os caminhos com revisão pendente e sem pendência, garantindo a mesma quantidade e ordem de outputs.

### Exercício 32 — Conectar envio de mensagem ao agente

**Reutiliza o esqueleto Gradio e agente previamente construído.** Implemente `send_message(message, history, thread_id, pending)`: impeça novo envio enquanto houver pendências, trate texto vazio, invoque `agent.invoke` com a configuração da thread e atualize o histórico e os componentes retornados.

### Exercício 33 — Exibir as ações propostas no painel

**Reutiliza `process_result` e estado Gradio.** Faça o callback interpretar `result.interrupts`, guardar IDs e valores em `gr.State`, mostrar o nome da tool, os argumentos em JSON formatado e o número de pendências, e tornar o painel visível.

### Exercício 34 — Centralizar retomada do agente

**Reutiliza agente, thread e callbacks.** Implemente `resume_agent(decision, history, pending, thread_id)`. Valide que existe pendência, use o primeiro ID, chame `agent.invoke(Command(...))` com o mesmo contexto/configuração e passe o resultado novamente ao processador da interface.

### Exercício 35 — Conectar callbacks de aprovação e rejeição

**Reutiliza `resume_agent`.** Implemente `approve_action` e `reject_action`, conecte cada função ao botão correspondente e confira a correspondência entre parâmetros recebidos de `inputs` e argumentos do callback. A rejeição deve usar motivo digitado ou mensagem padrão.

### Exercício 36 — Validar edição de argumentos antes do resume

**Reutiliza fluxo HITL e Gradio.** Implemente `edit_action` para validar JSON, exigir um objeto, validar os campos permitidos para a tool e verificar tipos, strings vazias e valores numéricos. Para entrada inválida, mostre `gr.Error` sem retomar o agente.

## 7. Integração, erros, depuração e boas práticas

### Exercício 37 — Proteger a interface contra concorrência de revisão

**Reutiliza interface integrada.** Implemente uma regra de controle que bloqueie novos envios durante uma revisão pendente e só libere o campo de mensagem quando a execução não retornar novas interrupções. Explique por que esconder o painel sem resolver o checkpoint não basta.

### Exercício 38 — Ordenar HITL, retry e tratamento de erros

**Independente.** Monte a lista de middlewares para um agente com HITL por chamada e retry de tool. Escreva uma validação da ordem e explique como posicionar a barreira humana para evitar nova aprovação indevida durante retries.

### Exercício 39 — Garantir segurança de efeitos colaterais

**Independente.** Revise um fluxo hipotético que cobra uma operação dentro de um middleware anterior ao HITL. Reestruture-o para que o efeito ocorra somente depois da aprovação e implemente uma chave de idempotência para proteger contra repetição.

### Exercício 40 — Depurar falha de integração por camada

**Independente.** Escreva uma função que classifique falhas hipotéticas em configuração do modelo/API, escolha de tool pelo modelo, validação de argumentos, execução da tool, checkpoint/resume ou mapeamento Gradio. Para cada classe, indique os dados e logs que inspecionaria sem expor segredos.

### Exercício 41 — Recuperar revisão após atualização da página

**Independente.** Implemente o desenho de uma rotina de recuperação que, ao receber um `thread_id`, consulte o estado salvo do agente e reconstrua pendências para a interface. Diferencie o estado efêmero de `gr.State` e o estado persistido pelo checkpointer; considere persistência em memória versus backend durável.

### Exercício 42 — Construir o agente integrado

**Desafio integrador; reutilize componentes dos exercícios anteriores.** Monte um agente com tool comum, tool sensível simulada, tools de memória longa, `Context`, `ToolRuntime`, checkpointer, Store, logging em middleware e HITL `per_call`. Registre cada componente no ponto correto de `create_agent` e justifique o caminho de dados entre eles.

### Exercício 43 — Integrar aplicação Gradio com retomada recursiva

**Desafio integrador; reutilize o agente anterior e os callbacks.** Construa a interface com chatbot, thread, pendências, envio, painel ocultável e botões aprovar/editar/rejeitar. O processador deve lidar com resposta final, múltiplas pendências em sequência e novo interrupt após `Command(resume=...)`, mantendo corretas as saídas de cada callback.

### Exercício 44 — Criar plano de validação ponta a ponta

**Desafio integrador; reutilize a aplicação completa.** Escreva verificações para tool comum e `ToolMessage`, memória curta na mesma thread, memória longa entre threads do mesmo usuário, aprovação, edição, rejeição, múltiplas tool calls, erro de JSON, proteção contra novo envio e retomada. Registre separadamente o que pode ser validado sem modelo real e o que depende de comportamento do modelo/API.